In [ ]:
import openai
import json
# OpenAI API 키 설정 (YOUR_API_KEY를 실제 키로 대체)
openai.api_key = '<REMOVED: set OPENAI_API_KEY env var>' #API key

def QA_raw2list (QAlist,QAraw,summary):
    QArawSET=QAraw.split('\n\n')
    for i in range(len(QArawSET)):
        Q=QArawSET[i].split('Answer:')[0].replace('Question:','')
        A=QArawSET[i].split('Answer:')[1]
        QAlist.append({'Question' : Q, 'Answer' : A,'summary' : summary })

QAlist=[]
for i in range(0,1000):
# for i in [200]:
    try:
        # 생성할 텍스트 입력
        prompt1 = f"""제목: {raw_data[i]['Title']}\n
                    날짜: {raw_data[i]['Time']}\n 
                    내용: {raw_data[i]['Content'][:(len(raw_data[i]['Content'])//4)]}
                        """
        prompt2 = f"""위 내용에 이어지는 내용입니다.\n
                        {raw_data[i]['Content'][(len(raw_data[i]['Content'])//4):2*(len(raw_data[i]['Content'])//4)]}"""
        prompt21 = f"""위 내용에 이어지는 내용입니다.\n
                        {raw_data[i]['Content'][2*(len(raw_data[i]['Content'])//4):3*(len(raw_data[i]['Content'])//4)]}"""
        prompt22 = f"""위 내용에 이어지는 내용입니다.\n
                        {raw_data[i]['Content'][3*(len(raw_data[i]['Content'])//4):4*(len(raw_data[i]['Content'])//4)]}"""
                        
        prompt3 =  f"""위 입력되었던 내용을 가지고 
        1. 악성코드 설명,
        2. 악성코드의 공격 방법,
        3. 악성코드의 공격 흐름
        4. 악성코드의 감염된 pc의 증상,
        5. 악성 코드의 파일 진단 리스트(파일 진단 리스트: 리스트를 나열하여 한 문장으로 답변 할것),
        6. 악성 코드의 파일 행위 리스트(파일 행위 리스트: 리스트를 나열하여 한 문장으로 답변 할것),
        7. 마지막에 IOC 리스트를 작성해줘 (IOC 리스트: 리스트를 나열하여 한 문장으로 답변 할것)).
        """

        response = openai.ChatCompletion.create(model="gpt-3.5-turbo-1106",
                                                messages=[{"role": "user", "content": prompt1},
                                                        {"role": "user", "content": prompt2},
                                                        {"role": "user", "content": prompt21},
                                                        {"role": "user", "content": prompt22},
                                                        {"role": "user", "content": prompt3}
                                                        ]
                                                )
        summary =f"""제목: {raw_data[i]['Title']}\n 날짜: {raw_data[0]['Time']}\n 내용: {response['choices'][0]['message']['content']}\n"""
        
        prompt4 =  f"""{summary}
    Based on the text above, create a malware-related question and answer pair.
    The requirements are as follows
    1. generate question content and corresponding answer content.
    2. Ensure that the output of the question and the answer is "Question: (Question content)", "Answer: (Answer content)".
    3. Questions must be written in Korean.
    4. Create answer pairs for the following questions.
    5. Write your answer in 2-3 sentences. 
    5. No more than 100 words is ideal.
    6. Questions and answers should be separated by starting with "Question:" and "Answer:" Please make sure to follow this format.
    7. If the question contains one of the following: "File Diagnostics List", "Action Diagnostics List" and "IOC List", please list the answers so that the list is not omitted.
    8. Understand that similar contents such as “information leakage,” “file corruption,” “remote manipulation,” “file deletion,” “information collection,” “ad insertion,” and “obtaining administrator privileges” are also attacks.
    9. Don't create question-answer pairs that use the same words.
    10. Create 10 question-answer pairs.
    11. Separate each question-answer pair with '\n\n'
    """
        # 9. Half of them, create another sentence with duplicate content.

        
        response_QA = openai.ChatCompletion.create(model="gpt-4o", messages=[{"role": "user", "content": prompt4}])
        QA = response_QA['choices'][0]['message']['content']
        QA_raw2list(QAlist,QA, summary)
    except Exception as e:
        print(f"An error occurred in iteration {i}: \n{e}")
        continue  # 에러가 발생해도 계속 다음 반복을 진행
    
    
#List2JSON
json_data = json.dumps(QAlist, ensure_ascii=False, indent=4)
with open('QA.json', 'w', encoding='utf-8') as json_file:
    json_file.write(json_data)

  
#10script에 5m 25.7s
#10script에 6m 25.7s

file_path = "./QA.json"

with open(file_path, 'r') as file:
    json_data = json.load(file)


for i in range(len(json_data)):
    re= json_data[i]['summary'].split('\n')[0][4:]
    json_data[i]['Question']=json_data[i]['Question'].replace('\n','').replace(' 주어진',re+'에 대한')


a= json.dumps(json_data, ensure_ascii=False, indent=4)
with open('QA.json', 'w', encoding='utf-8') as json_file:
    json_file.write(a)